# IntegriDoc Phase 6: VLM Integration & Explainability

In this notebook, we combine OCR (`pytesseract`) with a Vision-Language Model (`Qwen2.5-VL-3B-Instruct`) to act as a **Forensic Evidence Interpreter**.

## 1. Setup Environment
First, we clone the code, install `tesseract-ocr` on the system, and install our python dependencies (including `transformers` and `qwen-vl-utils`).

In [ ]:
!git clone https://github.com/ivaaneoski/IntegriDoc.git
%cd IntegriDoc
!apt-get install -y tesseract-ocr
!pip install -e .


## 2. Generate a Test Document
We need a fake document to analyze.

In [ ]:
!python scripts/generate_dataset.py --num_sources 10


## 3. Run the OCR Extractor
Let's extract the text and bounding boxes from a generated document.

In [ ]:
import json
from src.ocr.extractor import extract_text_regions

# Pick a generated tampered document
with open('data/manifests/val.json', 'r') as f:
    manifest = json.load(f)
    
# Find a tampered image
tampered_record = next(r for r in manifest if r['label'] == 'TAMPERED')
image_path = tampered_record['image_path']

ocr_data = extract_text_regions(image_path)
print(f"Extracted {len(ocr_data)} text regions.")
print(json.dumps(ocr_data[:2], indent=2)) # Print first 2 regions


## 4. Load the VLM and Interpret Evidence
Now we pass the document image, the OCR data, and a mock classifier score to Qwen2.5-VL to see if it can generate a structured forensic report.

In [ ]:
from src.vlm.qwen_interpreter import ForensicVLM

vlm = ForensicVLM()

# Mock a classifier score of 0.95 (High Confidence Forgery)
report = vlm.analyze_document(
    image_path=image_path, 
    ocr_data=ocr_data, 
    classifier_score=0.95, 
    has_heatmap=False
)

print("\n=== FINAL VLM REPORT ===")
print(json.dumps(report, indent=2))
